In [1]:
pip install openpyxl --upgrade

  Attempting uninstall: openpyxl
    Found existing installation: openpyxl 3.0.9
    Uninstalling openpyxl-3.0.9:
      Successfully uninstalled openpyxl-3.0.9
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Retail Data Engineering – End-to-End Pipeline
## ABC Retail Solutions | NeoStats Case Study

In this notebook, I am working with a multi-sheet retail dataset to perform a complete
data engineering and analytics workflow.

The goal is to clean the data, handle inconsistencies, mask PII, enrich and transform
the data, calculate business KPIs, and finally export a clean dataset ready for Power BI.

This includes:
- Understanding the structure of the retail dataset
- Handling missing values, duplicates, and data quality issues
- Standardising categories, product names, and date formats
- **Masking PII** (email and phone) using SHA-256 hashing
- Calculating revenue and extracting date-based features
- Aggregating business KPIs (Total Revenue, Revenue by Category, City, etc.)
- Exporting all results as CSV files for Power BI dashboards

The overall idea is to simulate a real-world data engineering + analytics workflow.

## Importing Required Libraries

First, I am importing all the necessary Python libraries that will be used throughout
the notebook.

- **pandas & numpy** → for data handling and numerical operations
- **hashlib** → for PII masking (SHA-256 hashing)
- **os & logging** → for output management and structured logging
- **warnings** → to suppress unnecessary warnings

Keeping everything in one place helps in maintaining clarity and avoiding repetition later.

In [3]:
import pandas as pd
import numpy as np
import hashlib
import logging
import os
import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Loading the Dataset

The dataset is stored in an Excel file containing three sheets:

- **product_details** – Dimension table with product IDs, names, categories, and standard prices (10 records)
- **retail_data1** – First transaction dataset (4,243 records)
- **retail_data2** – Second transaction dataset (4,251 records)

I am loading each sheet separately so that I can inspect and process them individually
before combining. The two retail datasets are then concatenated into a single DataFrame
for unified processing.

In [4]:
# ── UPDATE THIS PATH to your local file location ──────────────────────────────
EXCEL_FILE = r"C:\Users\mishr\Desktop\Neo Stat\USECASE - Data Engineering.xlsx"   # ← change if needed
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load all 3 sheets
product_details = pd.read_excel(EXCEL_FILE, sheet_name="product_details")
retail_data1    = pd.read_excel(EXCEL_FILE, sheet_name="retail_data1")
retail_data2    = pd.read_excel(EXCEL_FILE, sheet_name="retail_data2")

print(f"✅ product_details  : {product_details.shape}")
print(f"✅ retail_data1     : {retail_data1.shape}")
print(f"✅ retail_data2     : {retail_data2.shape}")

✅ product_details  : (10, 4)
✅ retail_data1     : (4243, 16)
✅ retail_data2     : (4251, 16)


## Initial Data Exploration

Before doing any transformations, it's important to understand what the data looks like.

Here I am:
- Checking column names across all three sheets
- Viewing sample records from each dataset
- Getting a basic idea of data structure and types

This step helps in identifying inconsistencies and planning the cleaning process.

In [5]:
print("=== PRODUCT DETAILS COLUMNS ===")
print(product_details.columns.tolist())
print(product_details.head(3))

print("\n=== RETAIL DATA1 COLUMNS ===")
print(retail_data1.columns.tolist())
print(retail_data1.head(3))

print("\n=== RETAIL DATA2 COLUMNS ===")
print(retail_data2.columns.tolist())
print(retail_data2.head(3))

=== PRODUCT DETAILS COLUMNS ===
['product_id', 'product_name', 'category', 'price']
   product_id product_name     category   price
0         101       Laptop  Electronics  250000
1         102        Phone  Electronics   70000
2         103        Shirt     Clothing    1300

=== RETAIL DATA1 COLUMNS ===
['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price', 'product_name', 'category', 'purchase_location', 'city', 'transaction_date', 'quantity', 'payment_method', 'discount', 'email', 'phone', 'payment_status']
   transaction_id  customer_id   customer_name  product_id    price  \
0               1          642   Troy Mitchell         108  15000.0   
1               2          881  Sarah Guerrero         102  70000.0   
2               3          505   Samantha Hull         109  65000.0   

    product_name         category purchase_location       city  \
0  Mixer grinder  Home Appliances           offline  Bangalore   
1          Phone             ELEC            onl

## Tagging Source and Combining Datasets

Before merging, I am:
- Adding a **source** column to each dataset to track whether a record came from
  retail_data1 or retail_data2
- Concatenating both datasets into one unified DataFrame

This is a common pattern in data engineering — combining multiple data feeds into a
single table for unified analysis.

In [6]:
retail_data1["source"] = "retail_data1"
retail_data2["source"] = "retail_data2"

df = pd.concat([retail_data1, retail_data2], ignore_index=True)
print(f"✅ Combined dataset  : {df.shape[0]} rows, {df.shape[1]} cols (before cleaning)")

✅ Combined dataset  : 8494 rows, 17 cols (before cleaning)


## Checking for Missing Values

Missing data can cause issues in analysis and revenue calculations.

Here I am checking null values across the combined dataset.

This gives a clear idea of:
- Which columns have missing data
- How severe the issue is

Based on this, I will decide whether to fill values (from product_details) or drop rows.

In [7]:
print("=== MISSING VALUES – Combined Dataset ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

=== MISSING VALUES – Combined Dataset ===
price    809
dtype: int64


## Data Cleaning & Transformation

In this step, I am performing several cleaning operations on the combined dataset:

1. **Remove duplicate rows** – both transaction datasets may share exact duplicate entries
2. **Filter failed payments** – only successful transactions are relevant for revenue analysis
3. **Standardise column names** – strip spaces, lowercase, replace spaces with underscores
4. **Fix date formats** – the date column mixes Excel serial numbers and string formats (MM-DD-YYYY)
5. **Fix quantity** – drop rows where quantity ≤ 0 (invalid or zero-quantity orders)
6. **Fill missing prices** – use the product_details dimension table to fill in missing prices
7. **Standardise categories** – map shorthand codes (ELEC, FURN, CLOTH, HOME) to full names
8. **Standardise product names** – override with product_details as the authoritative source
9. **Standardise purchase_location, payment_method, city** – fix casing inconsistencies
10. **Validate discounts** – clip to valid range [0, 1]

Each step ensures the data is reliable and consistent for analysis.

In [8]:
# ── 2a. Remove exact duplicate rows ──────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Exact duplicate rows removed : {before - len(df)}")

# ── 2b. Keep only 'successful' transactions ───────────────────────────────────
before = len(df)
df = df[df["payment_status"].str.lower() == "successful"].copy()
print(f"Failed/invalid payments dropped : {before - len(df)}")

# ── 2c. Standardise column names ─────────────────────────────────────────────
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ── 2d. Fix date column (mixed formats) ───────────────────────────────────────
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    if isinstance(val, (int, float)):
        try:
            return pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(val))
        except Exception:
            return pd.NaT
    if isinstance(val, str):
        for fmt in ("%m-%d-%Y", "%Y-%m-%d", "%d-%m-%Y"):
            try:
                return pd.to_datetime(val, format=fmt)
            except ValueError:
                continue
        return pd.NaT
    return pd.Timestamp(val)

df["transaction_date"] = df["transaction_date"].apply(parse_date)
print(f"Rows with unparseable dates (set to NaT) : {df['transaction_date'].isna().sum()}")

# ── 2e. Fix quantity – drop invalid rows ──────────────────────────────────────
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
before = len(df)
df = df[df["quantity"] > 0].copy()
print(f"Rows with invalid quantity (≤ 0 or null) removed : {before - len(df)}")

# ── 2f. Fill missing prices from product_details ─────────────────────────────
product_details.columns = product_details.columns.str.strip().str.lower()
price_map = product_details.set_index("product_id")["price"].to_dict()
df["price"] = pd.to_numeric(df["price"], errors="coerce")
missing_before = df["price"].isna().sum()
df["price"] = df.apply(
    lambda row: price_map.get(row["product_id"], np.nan) if pd.isna(row["price"]) else row["price"],
    axis=1,
)
print(f"Missing prices filled from product_details : {missing_before - df['price'].isna().sum()}")
before = len(df)
df.dropna(subset=["price"], inplace=True)
print(f"Rows with unresolvable price dropped : {before - len(df)}")

# ── 2g. Standardise category ─────────────────────────────────────────────────
category_map = {
    "elec": "Electronics", "electronics": "Electronics",
    "cloth": "Clothing",    "clothing": "Clothing",
    "furn": "Furniture",    "furniture": "Furniture",
    "home appliances": "Home Appliances", "home": "Home Appliances",
}
df["category"] = df["category"].str.strip().str.lower().map(category_map).fillna(
    df["category"].str.strip().str.title()
)
cat_map = product_details.set_index("product_id")["category"].to_dict()
df["category"] = df["product_id"].map(cat_map).fillna(df["category"])
print("Category values standardised.")

# ── 2h. Standardise product_name ─────────────────────────────────────────────
name_map = product_details.set_index("product_id")["product_name"].to_dict()
df["product_name"] = df["product_id"].map(name_map).fillna(df["product_name"].str.strip().str.title())
print("Product names standardised.")

# ── 2i–2l. Standardise other columns ─────────────────────────────────────────
df["purchase_location"] = df["purchase_location"].str.strip().str.lower()
df["payment_method"]    = df["payment_method"].str.strip().str.title()
df["city"]              = df["city"].str.strip().str.title()
df["discount"]          = pd.to_numeric(df["discount"], errors="coerce").fillna(0).clip(0, 1)

df.reset_index(drop=True, inplace=True)
print(f"\n✅ Clean dataset size : {df.shape[0]} rows, {df.shape[1]} cols")

Exact duplicate rows removed : 0
Failed/invalid payments dropped : 494
Rows with unparseable dates (set to NaT) : 0
Rows with invalid quantity (≤ 0 or null) removed : 86
Missing prices filled from product_details : 801
Rows with unresolvable price dropped : 0
Category values standardised.
Product names standardised.

✅ Clean dataset size : 7914 rows, 17 cols


## Verifying Null Values After Cleaning

After applying all cleaning steps, I am verifying that there are no remaining null
values in the critical columns.

This confirms that the cleaning process was effective.

In [9]:
print("=== NULL VALUES AFTER CLEANING ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "✅ No null values in critical columns.")

=== NULL VALUES AFTER CLEANING ===
✅ No null values in critical columns.


## PII Masking – Email and Phone

A key requirement of this assignment is to **mask personally identifiable information (PII)**.

Customer email addresses and phone numbers must be protected before the data is used
for analysis or exported.

I am using **SHA-256 hashing** to:
- Irreversibly transform email and phone values
- Create consistent masked values (same input → same hash)
- Replace the original PII columns with hashed versions

The original `email` and `phone` columns are then **deleted** from the dataset.

In [10]:
def sha256_mask(value):
    """SHA-256 hash for PII masking. Returns empty string for null values."""
    if pd.isna(value) or str(value).strip() == "":
        return ""
    return hashlib.sha256(str(value).strip().lower().encode()).hexdigest()

df["email_masked"] = df["email"].apply(sha256_mask)
df["phone_masked"] = df["phone"].apply(sha256_mask)

# Drop original PII columns
df.drop(columns=["email", "phone"], inplace=True)
print("✅ PII columns (email, phone) hashed with SHA-256 and originals removed.")
print(f"   Sample masked email : {df['email_masked'].iloc[0][:30]}...")
print(f"   Sample masked phone : {df['phone_masked'].iloc[0][:30]}...")

✅ PII columns (email, phone) hashed with SHA-256 and originals removed.
   Sample masked email : 401f91ea12377426d55aed01003478...
   Sample masked phone : f896c10e8ef446251b8c01b70df1b3...


## Feature Engineering

To make the analysis more meaningful, I am creating new derived columns:

- **Revenue** = `price × quantity × (1 − discount)` — the actual revenue generated per transaction
- **Year, Month, Month Name** — extracted from transaction date for time-based analysis
- **Year-Month (Period)** — for monthly trend charts
- **Quarter** — for quarterly aggregation

These derived features are the foundation for all KPI calculations and Power BI visuals.

In [11]:
# Revenue calculation
df["revenue"] = (df["price"] * df["quantity"] * (1 - df["discount"])).round(2)

# Date parts
df["year"]       = df["transaction_date"].dt.year
df["month"]      = df["transaction_date"].dt.month
df["month_name"] = df["transaction_date"].dt.strftime("%B")
df["year_month"] = df["transaction_date"].dt.to_period("M").astype(str)
df["quarter"]    = df["transaction_date"].dt.quarter.apply(lambda q: f"Q{q}")

print("✅ Revenue calculated and date parts extracted.")
print(df[["transaction_id", "product_name", "price", "quantity", "discount", "revenue", "year_month"]].head(5))

✅ Revenue calculated and date parts extracted.
   transaction_id   product_name    price  quantity  discount   revenue  \
0               1  Mixer Grinder  15000.0         3      0.10   40500.0   
1               2          Phone  70000.0         5      0.35  227500.0   
2               3   Refrigerator  65000.0         2      0.05  123500.0   
3               4           Sofa  80000.0         2      0.05  152000.0   
4               5  Mixer Grinder  15000.0         1      0.05   14250.0   

  year_month  
0    2025-12  
1    2025-07  
2    2025-07  
3    2025-10  
4    2025-12  


## Enriching with Product Dimension Table

Now I am merging the clean transaction data with the `product_details` dimension table
using a **left join** on `product_id`.

This adds the `standard_price` from the product catalogue to the transaction records,
which is useful for comparing actual transaction prices against the standard price.

In [12]:
product_details_renamed = product_details.rename(columns={"price": "standard_price"})
df = df.merge(
    product_details_renamed[["product_id", "standard_price"]],
    on="product_id", how="left"
)
print(f"✅ Merged standard_price from product_details.")
print(f"   Final dataset shape : {df.shape}")

✅ Merged standard_price from product_details.
   Final dataset shape : (7914, 24)


## Exporting the Clean Dataset

After all processing and transformation, I am saving the fully cleaned and enriched
dataset as a CSV file.

This file can be used for:
- **Dashboard creation** (Power BI / Tableau)
- **Further analysis** and modelling
- **Reporting** and documentation

Exporting the final dataset ensures reusability of all the work done above.

In [13]:
CLEAN_CSV = os.path.join(OUTPUT_DIR, "retail_clean.csv")
df.to_csv(CLEAN_CSV, index=False)
print(f"✅ Clean dataset saved → {CLEAN_CSV}")
print(f"   Shape    : {df.shape}")
print(f"   Columns  : {df.columns.tolist()}")

✅ Clean dataset saved → output\retail_clean.csv
   Shape    : (7914, 24)
   Columns  : ['transaction_id', 'customer_id', 'customer_name', 'product_id', 'price', 'product_name', 'category', 'purchase_location', 'city', 'transaction_date', 'quantity', 'payment_method', 'discount', 'payment_status', 'source', 'email_masked', 'phone_masked', 'revenue', 'year', 'month', 'month_name', 'year_month', 'quarter', 'standard_price']


## Business KPIs & Aggregations

Here I am calculating the key business metrics required by the assignment:

- **Total Revenue** – overall revenue across all transactions
- **Total Transactions** – count of successful transaction IDs
- **Total Units Sold** – total quantity across all orders
- **Average Order Value** – mean revenue per transaction
- **Average Discount** – average discount percentage applied
- **Top Category** – category generating the highest revenue
- **Top City** – city generating the highest revenue
- **Top Product** – product generating the highest revenue

These KPIs provide a quick executive-level summary of the business performance.

In [14]:
total_revenue      = df["revenue"].sum()
total_transactions = df["transaction_id"].nunique()
total_units_sold   = df["quantity"].sum()
avg_order_value    = df["revenue"].mean()
avg_discount       = df["discount"].mean() * 100
top_category       = df.groupby("category")["revenue"].sum().idxmax()
top_city           = df.groupby("city")["revenue"].sum().idxmax()
top_product        = df.groupby("product_name")["revenue"].sum().idxmax()

print("=" * 50)
print("         📊 KEY PERFORMANCE INDICATORS")
print("=" * 50)
print(f"  Total Revenue         : ₹{total_revenue:,.2f}")
print(f"  Total Transactions    : {total_transactions}")
print(f"  Total Units Sold      : {int(total_units_sold)}")
print(f"  Avg Order Value       : ₹{avg_order_value:,.2f}")
print(f"  Avg Discount          : {avg_discount:.2f}%")
print(f"  Top Category          : {top_category}")
print(f"  Top City              : {top_city}")
print(f"  Top Product           : {top_product}")
print("=" * 50)

         📊 KEY PERFORMANCE INDICATORS
  Total Revenue         : ₹1,163,993,285.00
  Total Transactions    : 7914
  Total Units Sold      : 23596
  Avg Order Value       : ₹147,080.27
  Avg Discount          : 22.43%
  Top Category          : Electronics
  Top City              : Chennai
  Top Product           : Laptop


## Revenue by Category

Here I am aggregating total revenue, order count, and units sold by product category.

This breakdown is essential for understanding which categories drive the most business
and for creating the **Category Trends** page in Power BI.

In [15]:
rev_cat = (
    df.groupby("category")
    .agg(
        total_revenue    = ("revenue",        "sum"),
        total_orders     = ("transaction_id", "count"),
        total_units      = ("quantity",       "sum"),
        avg_discount_pct = ("discount",       lambda x: round(x.mean() * 100, 2)),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
rev_cat["revenue_share_%"] = (rev_cat["total_revenue"] / rev_cat["total_revenue"].sum() * 100).round(2)
rev_cat.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_category.csv"), index=False)
print("✅ Revenue by Category saved.")
print(rev_cat.to_string(index=False))

✅ Revenue by Category saved.
       category  total_revenue  total_orders  total_units  avg_discount_pct  revenue_share_%
    Electronics    674462500.0          2395         7082             22.62            57.94
      Furniture    243413000.0          1581         4836             22.45            20.91
Home Appliances    229025250.0          2373         7076             22.19            19.68
       Clothing     17092535.0          1565         4602             22.50             1.47


## Revenue by City

Here I am aggregating total revenue by city.

This is used to create the **Regional Insights** page in Power BI — showing which
cities are the top contributors to revenue and identifying regional trends.

In [16]:
rev_city = (
    df.groupby("city")
    .agg(
        total_revenue = ("revenue",        "sum"),
        total_orders  = ("transaction_id", "count"),
        total_units   = ("quantity",       "sum"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
rev_city["revenue_share_%"] = (rev_city["total_revenue"] / rev_city["total_revenue"].sum() * 100).round(2)
rev_city.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_city.csv"), index=False)
print("✅ Revenue by City saved.")
print(rev_city.to_string(index=False))

✅ Revenue by City saved.
     city  total_revenue  total_orders  total_units  revenue_share_%
  Chennai    246028300.0          1579         4705            21.14
    Delhi    242022530.0          1636         4893            20.79
Hyderabad    232881480.0          1631         4816            20.01
   Mumbai    226340465.0          1620         4814            19.45
Bangalore    216720510.0          1448         4368            18.62


## Revenue by Product

Here I am aggregating total revenue, orders, and units sold at the individual product level.

This is used to create the **Product Performance** page in Power BI — showing which
specific products are the best and worst performers.

In [17]:
rev_prod = (
    df.groupby(["product_id", "product_name", "category"])
    .agg(
        total_revenue = ("revenue",        "sum"),
        total_orders  = ("transaction_id", "count"),
        total_units   = ("quantity",       "sum"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
rev_prod.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_product.csv"), index=False)
print("✅ Revenue by Product saved.")
print(rev_prod.to_string(index=False))

✅ Revenue by Product saved.
 product_id  product_name        category  total_revenue  total_orders  total_units
        101        Laptop     Electronics    466250000.0           824         2407
        106          Sofa       Furniture    150708000.0           791         2412
        102         Phone     Electronics    127568000.0           789         2353
        109  Refrigerator Home Appliances    117975000.0           763         2325
        107  Dining Table       Furniture     92705000.0           790         2424
        110     Microwave Home Appliances     83659500.0           801         2395
        105            TV     Electronics     80644500.0           782         2322
        108 Mixer Grinder Home Appliances     27390750.0           809         2356
        104         Shoes        Clothing     14864400.0           823         2400
        103         Shirt        Clothing      2228135.0           742         2202


## Revenue by Month

Here I am aggregating revenue on a monthly basis to identify trends over time.

This time-series data is used to create **trend line charts** in Power BI, showing
how revenue evolved month by month and identifying seasonal patterns.

In [18]:
rev_mon = (
    df.dropna(subset=["transaction_date"])
    .groupby(["year", "month", "month_name", "year_month", "quarter"])
    .agg(
        total_revenue = ("revenue",        "sum"),
        total_orders  = ("transaction_id", "count"),
    )
    .reset_index()
    .sort_values(["year", "month"])
)
rev_mon.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_month.csv"), index=False)
print("✅ Revenue by Month saved.")
print(rev_mon.to_string(index=False))

✅ Revenue by Month saved.
 year  month month_name year_month quarter  total_revenue  total_orders
 2025      1    January    2025-01      Q1      9342250.0            57
 2025      2   February    2025-02      Q1      6064440.0            47
 2025      3      March    2025-03      Q1      7949055.0            51
 2025      4      April    2025-04      Q2     57418535.0           395
 2025      5        May    2025-05      Q2     96635265.0           633
 2025      6       June    2025-06      Q2     92879965.0           644
 2025      7       July    2025-07      Q3     91364460.0           653
 2025      8     August    2025-08      Q3     99959810.0           665
 2025      9  September    2025-09      Q3     97668850.0           619
 2025     10    October    2025-10      Q4     96595110.0           634
 2025     11   November    2025-11      Q4     88707915.0           613
 2025     12   December    2025-12      Q4     86209270.0           655
 2026      1    January    2026-01    

## Revenue by Payment Method

Here I am analysing how different payment methods contribute to total revenue.

This helps understand customer payment preferences (Cash, Card, UPI, NetBanking)
and is used in the **Revenue Analysis** page of the Power BI dashboard.

In [19]:
rev_pay = (
    df.groupby("payment_method")
    .agg(
        total_revenue = ("revenue",        "sum"),
        total_orders  = ("transaction_id", "count"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
rev_pay["revenue_share_%"] = (rev_pay["total_revenue"] / rev_pay["total_revenue"].sum() * 100).round(2)
rev_pay.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_payment_method.csv"), index=False)
print("✅ Revenue by Payment Method saved.")
print(rev_pay.to_string(index=False))

✅ Revenue by Payment Method saved.
payment_method  total_revenue  total_orders  revenue_share_%
          Card    302321530.0          1994            25.97
          Cash    297436650.0          2003            25.55
           Upi    291428345.0          2016            25.04
    Netbanking    272806760.0          1901            23.44


## Revenue by Purchase Channel (Online vs Offline)

Here I am comparing revenue generated through online and offline channels.

This split is important for understanding where customers are shopping and
is used in the **Revenue Analysis** page of Power BI.

In [20]:
rev_ch = (
    df.groupby("purchase_location")
    .agg(
        total_revenue = ("revenue",        "sum"),
        total_orders  = ("transaction_id", "count"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
rev_ch["revenue_share_%"] = (rev_ch["total_revenue"] / rev_ch["total_revenue"].sum() * 100).round(2)
rev_ch.to_csv(os.path.join(OUTPUT_DIR, "revenue_by_channel.csv"), index=False)
print("✅ Revenue by Channel saved.")
print(rev_ch.to_string(index=False))

✅ Revenue by Channel saved.
purchase_location  total_revenue  total_orders  revenue_share_%
           online    584104340.0          3911            50.18
          offline    579888945.0          4003            49.82


## Pipeline Complete – Output Summary

All steps of the data engineering pipeline have been completed successfully.

**Output files created in the `output/` folder:**

| File | Purpose |
|---|---|
| `retail_clean.csv` | Main fact table for Power BI |
| `kpi_summary.csv` | KPI cards for the dashboard |
| `revenue_by_category.csv` | Category Trends page |
| `revenue_by_city.csv` | Regional Insights page |
| `revenue_by_product.csv` | Product Performance page |
| `revenue_by_month.csv` | Monthly Trend chart |
| `revenue_by_payment_method.csv` | Payment mix chart |
| `revenue_by_channel.csv` | Online vs Offline split |

**Next step:** Load `retail_clean.csv` (and the aggregation files) into Power BI
to build the interactive dashboard with the required pages:
- Revenue Analysis
- Product Performance
- Category Trends
- Regional Insights

In [25]:
import os

OUTPUT_DIR   = "output"
OUTPUT_EXCEL = os.path.join(OUTPUT_DIR, "ABC_Retail_Output.xlsx")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Output path set: {OUTPUT_EXCEL}")



log.info("=" * 60)
log.info("STEP 8: Exporting all outputs → single Excel file")
log.info("=" * 60)
 
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df.to_excel(writer,        sheet_name="retail_clean",             index=False)
    kpi_df.to_excel(writer,    sheet_name="kpi_summary",              index=False)
    rev_cat.to_excel(writer,   sheet_name="revenue_by_category",      index=False)
    rev_city.to_excel(writer,  sheet_name="revenue_by_city",          index=False)
    rev_prod.to_excel(writer,  sheet_name="revenue_by_product",       index=False)
    rev_mon.to_excel(writer,   sheet_name="revenue_by_month",         index=False)
    rev_pay.to_excel(writer,   sheet_name="revenue_by_payment",       index=False)
    rev_ch.to_excel(writer,    sheet_name="revenue_by_channel",       index=False)
 
log.info(f"✅ All outputs saved → {OUTPUT_EXCEL}")
log.info("   Sheets inside:")
log.info("     • retail_clean")
log.info("     • kpi_summary")
log.info("     • revenue_by_category")
log.info("     • revenue_by_city")
log.info("     • revenue_by_product")
log.info("     • revenue_by_month")
log.info("     • revenue_by_payment")
log.info("     • revenue_by_channel")
log.info("=" * 60)
 
print("\n✅ Pipeline complete! Output → output/ABC_Retail_Output.xlsx")
 

2026-05-31 11:55:36  [INFO]  ============================================================
2026-05-31 11:55:36  [INFO]  STEP 8: Exporting all outputs → single Excel file
2026-05-31 11:55:36  [INFO]  ============================================================


✅ Output path set: output\ABC_Retail_Output.xlsx


2026-05-31 11:55:42  [INFO]  ✅ All outputs saved → output\ABC_Retail_Output.xlsx
2026-05-31 11:55:42  [INFO]     Sheets inside:
2026-05-31 11:55:42  [INFO]       • retail_clean
2026-05-31 11:55:42  [INFO]       • kpi_summary
2026-05-31 11:55:42  [INFO]       • revenue_by_category
2026-05-31 11:55:42  [INFO]       • revenue_by_city
2026-05-31 11:55:42  [INFO]       • revenue_by_product
2026-05-31 11:55:42  [INFO]       • revenue_by_month
2026-05-31 11:55:42  [INFO]       • revenue_by_payment
2026-05-31 11:55:42  [INFO]       • revenue_by_channel
2026-05-31 11:55:42  [INFO]  ============================================================



✅ Pipeline complete! Output → output/ABC_Retail_Output.xlsx
